In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

IMAGE_PATH = PROJECT_ROOT / "images"
IMAGE_PATH.mkdir(parents=True, exist_ok=True)

print("Image folder ready:", IMAGE_PATH)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid")

PROJECT_ROOT = Path.cwd().parent
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "cleaned_transactions.csv"

df = pd.read_csv(DATA_PATH, parse_dates=["date"])

print("Dataset loaded:", df.shape)

In [ ]:
df.describe()

In [ ]:
#cell 3
print("Transaction types:")
display(df["txn_description"].value_counts())

print("\nTransaction movement:")
display(df["movement"].value_counts())

print("\nTransaction status:")
display(df["status"].value_counts())

In [ ]:
#cell 4
plt.figure(figsize=(10, 5))

sns.countplot(
    data=df,
    x="txn_description",
    order=df["txn_description"].value_counts().index
)

plt.title("Distribution of Transaction Types")
plt.xlabel("Transaction Type")
plt.ylabel("Number of Transactions")
plt.xticks(rotation=30)

plt.tight_layout()
plt.savefig(
    IMAGE_PATH / "transaction_type_distribution.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

In [ ]:
#cell 5
plt.figure(figsize=(10, 5))

sns.histplot(
    data=df,
    x="amount",
    bins=50,
    kde=True
)

plt.xscale("log")
plt.title("Distribution of Transaction Amounts (Log Scale)")
plt.xlabel("Transaction Amount (AUD)")
plt.ylabel("Frequency")

plt.tight_layout()
plt.savefig(
    IMAGE_PATH / "transaction_amount_distribution.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

In [ ]:
#cell 6
plt.figure(figsize=(7, 5))

sns.countplot(
    data=df,
    x="movement"
)

plt.title("Debit vs Credit Transactions")
plt.xlabel("Transaction Movement")
plt.ylabel("Number of Transactions")

plt.tight_layout()
plt.savefig(
    IMAGE_PATH / "debit_credit_distribution.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

In [ ]:
# cell 7
monthly_transactions = (
    df.groupby(df["date"].dt.to_period("M"))
      .size()
)

monthly_transactions.index = monthly_transactions.index.astype(str)

plt.figure(figsize=(9, 5))

monthly_transactions.plot(
    kind="line",
    marker="o",
    linewidth=2
)

plt.title("Monthly Transaction Trends")
plt.xlabel("Month")
plt.ylabel("Number of Transactions")
plt.xticks(rotation=30)
plt.grid(True)

plt.tight_layout()
plt.savefig(
    IMAGE_PATH / "monthly_transaction_trend.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

display(monthly_transactions)

### Observations from Transaction-Level Analysis

**1. Transaction Type Distribution**
- SALES-POS and POS are the most frequent transaction types.
- PHONE BANK transactions have the lowest frequency.
- POS-based activities account for a significant portion of the dataset.

**2. Transaction Amount Distribution**
- Transaction amounts are heavily right-skewed.
- Most transactions involve relatively small amounts.
- A few high-value transactions increase the overall mean.
- The mean (187.93 AUD) is considerably higher than the median (29 AUD).

**3. Debit vs Credit Analysis**
- Debit transactions account for approximately 92.7% of all records.
- Credit transactions account for approximately 7.3%.
- The dataset has a considerable imbalance between debit and credit transactions.

**4. Monthly Transaction Trends**
- Transaction volume increases slightly from August to October 2018.
- October records the highest number of transactions.
- The available three-month period is insufficient to establish long-term seasonal patterns.

## Customer-Level Analysis

This section examines transaction behaviour at the customer level,
including transaction frequency, debit and credit activity,
and differences in transaction amounts.

In [ ]:
#cell 8
unique_customers = df["customer_id"].nunique()

print("Total unique customers:", unique_customers)
print("Total transactions:", len(df))
print(
    "Average transactions per customer:",
    round(len(df) / unique_customers, 2)
)

In [ ]:
#cell 9
customer_frequency = (
    df.groupby("customer_id")
      .size()
      .sort_values(ascending=False)
)

print("Customer transaction frequency:")
display(customer_frequency.describe().round(2))

In [ ]:
# Visualisation
plt.figure(figsize=(10, 5))

sns.histplot(
    customer_frequency,
    bins=20,
    kde=True
)

plt.title("Distribution of Transactions per Customer")
plt.xlabel("Number of Transactions")
plt.ylabel("Number of Customers")
plt.tight_layout()
plt.savefig(
    IMAGE_PATH / "customer_transaction_frequency.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

In [ ]:
# cell 10
df["debit_amount"] = df["amount"].where(
    df["movement"] == "debit", 0
)

df["credit_amount"] = df["amount"].where(
    df["movement"] == "credit", 0
)

customer_summary = df.groupby("customer_id").agg(
    total_transactions=("amount", "count"),
    debit_count=("debit_amount", lambda x: (x > 0).sum()),
    credit_count=("credit_amount", lambda x: (x > 0).sum()),
    total_debit=("debit_amount", "sum"),
    total_credit=("credit_amount", "sum"),
    average_transaction=("amount", "mean")
)

customer_summary["average_debit"] = (
    customer_summary["total_debit"] / customer_summary["debit_count"]
)

customer_summary["average_credit"] = (
    customer_summary["total_credit"] / customer_summary["credit_count"]
)

customer_summary["net_flow"] = (
    customer_summary["total_credit"] - customer_summary["total_debit"]
)

display(customer_summary.describe().round(2))

In [ ]:
# cell 11
plt.figure(figsize=(10, 5))

sns.scatterplot(
    data=customer_summary,
    x="total_debit",
    y="total_credit"
)

plt.title("Customer Debit vs Credit Amounts")
plt.xlabel("Total Debit Amount (AUD)")
plt.ylabel("Total Credit Amount (AUD)")
plt.tight_layout()
plt.savefig(
    IMAGE_PATH / "debit_credit_amount_comparison.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

In [ ]:
# cell 12
movement_summary = df.groupby("movement")["amount"].agg(
    ["count", "sum", "mean", "median", "min", "max"]
)

display(movement_summary.round(2))

In [ ]:
# Visualzation
plt.figure(figsize=(8, 5))

sns.boxplot(
    data=df,
    x="movement",
    y="amount"
)

plt.yscale("log")
plt.title("Debit vs Credit Transaction Amounts")
plt.xlabel("Transaction Movement")
plt.ylabel("Transaction Amount (AUD, Log Scale)")
plt.tight_layout()
plt.savefig(
    IMAGE_PATH / "debit_vs_credit_amount_transaction.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

In [ ]:
# cell 13
transaction_type_summary = df.groupby("txn_description")["amount"].agg(
    ["count", "mean", "median", "min", "max"]
).sort_values("count", ascending=False)

display(transaction_type_summary.round(2))

In [ ]:
# Visualization
plt.figure(figsize=(11, 6))

sns.boxplot(
    data=df,
    x="txn_description",
    y="amount"
)

plt.yscale("log")
plt.title("Transaction Amounts Across Transaction Types")
plt.xlabel("Transaction Type")
plt.ylabel("Transaction Amount (AUD, Log Scale)")
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig(
    IMAGE_PATH / "transaction_amount_across_transaction_type.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

In [ ]:
# cell 14
gender_summary = df.groupby("gender").agg(
    transaction_count=("amount", "count"),
    average_amount=("amount", "mean"),
    median_amount=("amount", "median")
)

display(gender_summary.round(2))

In [ ]:
# Visualisation
plt.figure(figsize=(7, 5))

sns.countplot(data=df, x="gender")

plt.title("Transaction Distribution by Gender")
plt.xlabel("Gender")
plt.ylabel("Number of Transactions")
plt.tight_layout()
plt.savefig(
    IMAGE_PATH / "gender_distribution.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

In [ ]:
# cell 15
print("Age statistics:")
display(df["age"].describe().round(2))

In [ ]:
# Visualisation
plt.figure(figsize=(9, 5))

sns.histplot(
    data=df,
    x="age",
    bins=20,
    kde=True
)

plt.title("Age Distribution of Customers")
plt.xlabel("Age")
plt.ylabel("Frequency")
plt.tight_layout()
plt.savefig(
    IMAGE_PATH / "age_distribution.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

In [ ]:
# cell 16
df["age_group"] = pd.cut(
    df["age"],
    bins=[17, 25, 35, 45, 55, 100],
    labels=["18-25", "26-35", "36-45", "46-55", "56+"]
)

age_summary = df.groupby(
    "age_group", observed=True
)["amount"].agg(
    ["count", "mean", "median"]
)

display(age_summary.round(2))

In [ ]:
# Visualtisation
plt.figure(figsize=(9, 5))

sns.barplot(
    data=df,
    x="age_group",
    y="amount",
    errorbar=None
)

plt.title("Average Transaction Amount by Age Group")
plt.xlabel("Age Group")
plt.ylabel("Average Transaction Amount (AUD)")
plt.tight_layout()
plt.savefig(
    IMAGE_PATH / "age_group_comparison.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

In [ ]:
# cell 17
daily_transactions = (
    df.groupby(df["date"].dt.date)
      .size()
)

plt.figure(figsize=(14, 5))

daily_transactions.plot()

plt.title("Daily Transaction Volume")
plt.xlabel("Date")
plt.ylabel("Number of Transactions")
plt.tight_layout()
plt.savefig(
    IMAGE_PATH / "daily_transaction_volume.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

In [ ]:
# cell 18
daily_average_amount = (
    df.groupby(df["date"].dt.date)["amount"]
      .mean()
)

plt.figure(figsize=(14, 5))

daily_average_amount.plot()

plt.title("Daily Average Transaction Amount")
plt.xlabel("Date")
plt.ylabel("Average Transaction Amount (AUD)")
plt.tight_layout()
plt.savefig(
    IMAGE_PATH / "daily_average_amount.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

In [ ]:
# cell 19
# Ensure extraction is a datetime column
df["extraction"] = pd.to_datetime(
    df["extraction"],
    errors="coerce",
    utc=True
)

# Sort transactions chronologically for each customer
balance_analysis = df.sort_values(
    ["customer_id", "extraction"]
).copy()

# Calculate the difference between consecutive balances
balance_analysis["balance_change"] = (
    balance_analysis.groupby("customer_id")["balance"].diff()
)

# Calculate the expected balance change
balance_analysis["expected_change"] = np.where(
    balance_analysis["movement"] == "credit",
    balance_analysis["amount"],
    -balance_analysis["amount"]
)

# Compare actual and expected changes
balance_analysis["balance_difference"] = (
    balance_analysis["balance_change"]
    - balance_analysis["expected_change"]
)

display(
    balance_analysis[
        ["movement", "amount", "balance_change",
         "expected_change", "balance_difference"]
    ].head(10)
)

In [ ]:
# cell 20
valid_balance_records = balance_analysis.dropna(
    subset=["balance_difference"]
).copy()

valid_balance_records["balance_match"] = (
    valid_balance_records["balance_difference"].abs() < 0.01
)

match_count = valid_balance_records["balance_match"].sum()
mismatch_count = (~valid_balance_records["balance_match"]).sum()

match_percentage = (
    match_count / len(valid_balance_records) * 100
)

print("Valid balance comparisons:", len(valid_balance_records))
print("Matching records:", match_count)
print("Mismatching records:", mismatch_count)
print("Match percentage:", round(match_percentage, 2), "%")

In [ ]:
# cell 21
mismatch_records = valid_balance_records[
    ~valid_balance_records["balance_match"]
]

mismatch_by_type = (
    mismatch_records.groupby("txn_description")
    .size()
    .sort_values(ascending=False)
)

display(mismatch_by_type)

plt.figure(figsize=(10, 5))

mismatch_by_type.plot(kind="bar")

plt.title("Balance Mismatches by Transaction Type")
plt.xlabel("Transaction Type")
plt.ylabel("Number of Mismatches")
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig(
    IMAGE_PATH / "balance_mismatch_by_type.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

In [ ]:
# cell 22
numeric_df = df.select_dtypes(include=np.number)

# Remove identifiers and constant columns
numeric_df = numeric_df.drop(
    columns=[
        "customer_id",
        "transaction_id",
        "account",
        "merchant_code"
    ],
    errors="ignore"
)

correlation_matrix = numeric_df.corr()

display(correlation_matrix.round(2))

In [ ]:
# Visualisation
plt.figure(figsize=(10, 7))

sns.heatmap(
    correlation_matrix,
    annot=True,
    cmap="coolwarm",
    fmt=".2f",
    linewidths=0.5
)

plt.title("Correlation Heatmap of Numerical Variables")
plt.tight_layout()
plt.savefig(
    IMAGE_PATH / "correlation_heatmap.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

In [ ]:
# cell 23
missing_by_type = (
    df.groupby("txn_description")[
        ["card_present_flag", "merchant_id",
         "merchant_suburb", "merchant_state",
         "merchant_long_lat", "bpay_biller_code"]
    ]
    .apply(lambda group: group.isnull().mean() * 100)
)

display(missing_by_type.round(2))

In [ ]:
# cell 24
outlier_summary = []

for transaction_type, group in df.groupby("txn_description"):
    Q1 = group["amount"].quantile(0.25)
    Q3 = group["amount"].quantile(0.75)

    IQR = Q3 - Q1
    upper_limit = Q3 + (1.5 * IQR)

    outliers = group[group["amount"] > upper_limit]

    outlier_summary.append({
        "transaction_type": transaction_type,
        "upper_limit": round(upper_limit, 2),
        "potential_outliers": len(outliers),
        "percentage": round(
            len(outliers) / len(group) * 100, 2
        )
    })

outlier_summary = pd.DataFrame(outlier_summary)

display(
    outlier_summary.sort_values(
        "potential_outliers",
        ascending=False
    )
)

# Final Exploratory Data Analysis (EDA) Findings

## 1. Dataset Overview
- The dataset contains 12,043 transactions from 100 unique customers.
- It covers the period from August to October 2018.
- The dataset contains transaction details, customer information, merchant information and account balances.

## 2. Transaction Characteristics
- SALES-POS and POS are the most frequent transaction categories.
- Debit transactions account for approximately 92.7% of all records.
- Transaction amounts are heavily right-skewed, with a median of 29 AUD and a mean of approximately 187.93 AUD.

## 3. Customer Behaviour
- Customers have varying transaction frequencies and transaction amounts.
- Debit and credit activity differs considerably across the dataset.
- Credit transactions have substantially higher average values, largely due to salary payments.
- Customer-level analysis provides a better understanding of individual transaction behaviour.

## 4. Temporal Analysis
- Monthly transaction volume increases slightly from August to October.
- Daily transaction volume and average transaction amounts show variations across the observed period.
- Three months of data are insufficient to establish long-term trends or seasonality.

## 5. Balance Analysis
- Approximately 95.79% of consecutive balance changes match the expected change based on transaction amounts.
- A smaller group of records contains balance mismatches.
- These mismatches require further investigation and cannot automatically be classified as fraudulent transactions.

## 6. Potential Anomalies
- Transaction amounts contain unusually high values.
- IQR analysis helps identify statistically unusual transactions within individual transaction categories.
- Customer-level comparisons may help identify transactions that differ from an individual's usual behaviour.

## 7. Data Quality Considerations
- Merchant-related missing values are largely associated with transaction type.
- Constant and redundant columns should be considered during feature selection.
- The dataset contains no verified fraud labels.

## Overall Conclusion

The exploratory analysis reveals differences in transaction types, transaction values, customer behaviour and account balance patterns.

These findings provide a foundation for developing a customer transaction analytics and anomaly-detection system. Future analysis should focus on identifying unusual transaction behaviour relative to individual customers rather than treating every high-value transaction as suspicious.

Since the dataset covers only three months and does not contain verified fraud labels, the project should identify potential anomalies rather than claim confirmed fraud detection.

In [ ]:
# cell 26
print("Final EDA dataset shape:", df.shape)

print("\nDate range:")
print(df["date"].min(), "to", df["date"].max())

print("\nUnique customers:")
print(df["customer_id"].nunique())

print("\nTransaction types:")
display(df["txn_description"].value_counts())

print("\nMissing values:")
display(df.isnull().sum())